# Batch Inference — High Severity Claims

This script loads the **champion** model from Unity Catalog and runs batch inference on new/unscored claims.

**Workflow:**
1. Load the champion model via alias (`@champion`)
2. Read source claims data
3. Prepare features (matching the training pipeline)
4. Score all records
5. Write predictions to the scored predictions table

**Artefacts:**
- Model: `zurich_lcnc_rfp_catalog.pc_demo.high_severity_claim_model@champion`
- Source: `zurich_lcnc_rfp_catalog.pc_demo.curated_claims_analytics_designer`
- Output: `zurich_lcnc_rfp_catalog.pc_demo.claims_severity_predictions`

In [0]:
%pip install mlflow --quiet
dbutils.library.restartPython()

In [0]:
# ============================================================
# CONFIGURATION
# ============================================================

# Unity Catalog model (3-level namespace)
MODEL_NAME = "zurich_lcnc_rfp_catalog.pc_demo.high_severity_claim_model"
MODEL_ALIAS = "champion"  # Always fetch the current champion

# Tables
SOURCE_TABLE = "zurich_lcnc_rfp_catalog.pc_demo.curated_claims_analytics_designer"
PREDICTIONS_TABLE = "zurich_lcnc_rfp_catalog.pc_demo.claims_severity_predictions"

# Feature definitions (must match training pipeline)
CATEGORICAL_FEATURES = ["policy_type", "region"]
NUMERIC_FEATURES = [
    "premium_amount", "coverage_limit", "deductible",
    "risk_score", "reporting_delay_days", "claim_month", "claim_year",
]
ALL_FEATURES = CATEGORICAL_FEATURES + NUMERIC_FEATURES

print(f"Model:   {MODEL_NAME}@{MODEL_ALIAS}")
print(f"Source:  {SOURCE_TABLE}")
print(f"Output:  {PREDICTIONS_TABLE}")

In [0]:
# ============================================================
# LOAD CHAMPION MODEL FROM UNITY CATALOG
# ============================================================
%pip install mlflowimport mlflow


mlflow.set_registry_uri("databricks-uc")

model_uri = f"models:/{MODEL_NAME}@{MODEL_ALIAS}"
print(f"Loading model: {model_uri}")

champion_model = mlflow.pyfunc.load_model(model_uri)

# Show model metadata
model_info = mlflow.models.get_model_info(model_uri)
print(f"\n  Model URI:     {model_uri}")
print(f"  Run ID:        {model_info.run_id}")
print(f"  Model flavor:  {model_info.flavors.get('python_function', {}).get('loader_module', 'N/A')}")
print(f"  Signature:     {model_info.signature}")
print(f"\nChampion model loaded successfully.")

In [0]:
# ============================================================
# READ SOURCE CLAIMS DATA
# ============================================================
import pandas as pd
from datetime import datetime

df_source = spark.table(SOURCE_TABLE).toPandas()

print(f"Source claims loaded: {len(df_source):,} rows")
print(f"Columns: {list(df_source.columns)}")
print(f"\nSample (first 3 rows):")
display(df_source.head(3))

In [0]:
# ============================================================
# FEATURE PREPARATION
# ============================================================
# Apply the same transformations as the training pipeline:
#   - Select valid features (no leakage fields)
#   - One-hot encode categoricals
#   - Align columns with model's expected input schema

# Extract feature columns
df_features = df_source[ALL_FEATURES].copy()

# One-hot encode categoricals
df_encoded = pd.get_dummies(df_features, columns=CATEGORICAL_FEATURES, drop_first=False)

# Align with model's expected input columns
expected_cols = champion_model.metadata.get_input_schema().input_names()
missing_cols = set(expected_cols) - set(df_encoded.columns)
for col in missing_cols:
    df_encoded[col] = 0
df_encoded = df_encoded[expected_cols]

print(f"Feature matrix prepared: {df_encoded.shape[0]:,} rows x {df_encoded.shape[1]} features")
print(f"\nFeature columns: {list(df_encoded.columns[:10])}{'...' if len(df_encoded.columns) > 10 else ''}")

In [0]:
# ============================================================
# RUN BATCH INFERENCE
# ============================================================
from datetime import datetime

print(f"Scoring {len(df_encoded):,} claims with champion model...")

# Predict using the loaded champion model
predictions = champion_model.predict(df_encoded)

# Build predictions DataFrame
df_scored = pd.DataFrame({
    "claim_id": df_source["claim_id"].values,
    "policy_id": df_source["policy_id"].values,
    "policy_type": df_source["policy_type"].values,
    "region": df_source["region"].values,
    "risk_score": df_source["risk_score"].values,
    "premium_amount": df_source["premium_amount"].values,
    "coverage_limit": df_source["coverage_limit"].values,
    "deductible": df_source["deductible"].values,
    "predicted_high_severity": predictions.astype(int),
    "scored_at": datetime.now(),
    "model_version": MODEL_ALIAS,
})

# Add probability if model supports predict_proba
try:
    # For sklearn-backed models loaded via pyfunc, predict returns classes
    # Use the underlying sklearn model for probabilities
    import mlflow.sklearn
    sk_model = mlflow.sklearn.load_model(model_uri)
    probs = sk_model.predict_proba(df_encoded)[:, 1]
    df_scored["severity_probability"] = probs
    print(f"  Probabilities computed via sklearn predict_proba.")
except Exception:
    df_scored["severity_probability"] = None
    print(f"  Note: probability column set to None (model type may not support predict_proba).")

print(f"\nScoring complete!")
print(f"  Total predictions: {len(df_scored):,}")
print(f"  Predicted high severity: {(df_scored['predicted_high_severity'] == 1).sum():,} ({(df_scored['predicted_high_severity'] == 1).mean()*100:.1f}%)")
print(f"  Timestamp: {df_scored['scored_at'].iloc[0]}")
print(f"\nPreview:")
display(df_scored.head(5))

In [0]:
# ============================================================
# WRITE PREDICTIONS TO UNITY CATALOG
# ============================================================

df_scored_spark = spark.createDataFrame(df_scored)

# Append to predictions table (preserves historical scoring runs)
df_scored_spark.write.mode("append").saveAsTable(PREDICTIONS_TABLE)

print(f"Predictions written to: {PREDICTIONS_TABLE}")
print(f"  Rows appended: {len(df_scored):,}")
print(f"  Mode: append (historical runs preserved)")

# Verify
total_rows = spark.table(PREDICTIONS_TABLE).count()
print(f"\n  Total rows in predictions table: {total_rows:,}")
print(f"\nBatch inference complete.")

In [0]:
# ============================================================
# VALIDATION SUMMARY
# ============================================================

print("BATCH INFERENCE SUMMARY")
print("=" * 50)
print(f"  Model:          {MODEL_NAME}@{MODEL_ALIAS}")
print(f"  Source table:   {SOURCE_TABLE}")
print(f"  Output table:   {PREDICTIONS_TABLE}")
print(f"  Claims scored:  {len(df_scored):,}")
print(f"  High severity:  {(df_scored['predicted_high_severity'] == 1).sum():,}")
print(f"  Low severity:   {(df_scored['predicted_high_severity'] == 0).sum():,}")
print(f"  Scored at:      {df_scored['scored_at'].iloc[0]}")
print("=" * 50)
print("\nThis notebook can be scheduled as a Lakeflow Job for")
print("recurring batch scoring (e.g., daily or weekly).")
print("The champion alias ensures the latest promoted model is always used.")